# 01. Анализ датасета

Цель: понять состав собранных отзывов с Kaspi.

**Что проверяем:**
- Распределение по категориям
- Распределение длины (в словах и символах)
- Распределение рейтингов
- Распределение по языкам (русский / казахский / смешанный)
- Примеры отзывов из каждой категории


In [ ]:
# === Colab setup (skip if running locally) ===
import os, sys

IN_COLAB = "COLAB_GPU" in os.environ
if IN_COLAB:
    # Клонируем репо (первый раз) и переходим в него
    if not os.path.exists("/content/gan-reviews"):
        !git clone https://github.com/USER/gan-reviews.git /content/gan-reviews
    %cd /content/gan-reviews
    !pip install -q -r requirements-trainer.txt

# Добавляем корень репо в sys.path, чтобы импорты работали из notebooks/
sys.path.insert(0, os.path.abspath(".."))

# Mount Google Drive для сохранения чекпоинтов (опционально)
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    CHECKPOINT_DIR = "/content/drive/MyDrive/gan-reviews/checkpoints"
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
else:
    CHECKPOINT_DIR = "../checkpoints"
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print("CHECKPOINT_DIR =", CHECKPOINT_DIR)


In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

DATA_PATH = '../data/raw/reviews.jsonl'

rows = []
with open(DATA_PATH, encoding='utf-8') as f:
    for line in f:
        rows.append(json.loads(line))
df = pd.DataFrame(rows)
print(f'Всего отзывов: {len(df)}')
df.head()


## Распределение по категориям


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
df['category'].value_counts().plot(kind='bar', ax=ax)
ax.set_title('Reviews per category')
ax.set_ylabel('count')
plt.tight_layout()
plt.savefig('../results/eda_categories.png', dpi=120)
plt.show()


## Распределение длины


In [ ]:
df['n_words'] = df['text'].str.split().str.len()
df['n_chars'] = df['text'].str.len()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df['n_words'].hist(bins=40, ax=axes[0])
axes[0].set_title('Words per review')
df['n_chars'].hist(bins=40, ax=axes[1])
axes[1].set_title('Chars per review')
plt.tight_layout()
plt.savefig('../results/eda_lengths.png', dpi=120)
plt.show()


## Распределение рейтингов


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
df['rating'].value_counts().sort_index().plot(kind='bar', ax=ax)
ax.set_title('Rating distribution')
plt.savefig('../results/eda_ratings.png', dpi=120)
plt.show()

# Crosstab category x rating
ct = pd.crosstab(df['category'], df['rating'])
print(ct)


## Детекция языка

Простой подход: считаем долю символов казахского алфавита (ә, ө, ұ, ү, ң, қ, ғ, һ, і).
Точнее можно через `langdetect`, но эта эвристика быстра и достаточна.


In [ ]:
KZ_LETTERS = set('әөұүңқғһі')

def kz_ratio(text):
    if not text: return 0.0
    cyrillic = [c for c in text.lower() if c.isalpha()]
    if not cyrillic: return 0.0
    return sum(c in KZ_LETTERS for c in cyrillic) / len(cyrillic)

df['kz_ratio'] = df['text'].apply(kz_ratio)
df['lang'] = df['kz_ratio'].apply(
    lambda r: 'kk' if r > 0.05 else ('ru' if r >= 0 else 'unk')
)
print(df['lang'].value_counts())


## Примеры отзывов по категориям


In [ ]:
for cat in df['category'].unique():
    sub = df[df['category'] == cat].head(3)
    print(f'\n=== {cat} ===')
    for _, r in sub.iterrows():
        print(f"  [{r['rating']}*] {r['text'][:120]}")
